# MLIP Active-Learning Tutorial

This tutorial shows how the ALF `MLIPModel` (a MACE machine-learned interatomic
potential) is used in an **online active-learning loop**. Our use case: starting from a
pretrained foundation model and **finetuning** it into an accurate force field for a single
organic molecule, while spending as few expensive labels as possible.

### How this differs from the design tutorials

The protein-design tutorials *maximize a fitness*. Here we do the opposite kind of active
learning: we **minimize model error using as few expensive labels as possible**. Each label is a
quantum-chemistry calculation (DFT in production; here a fast GFN2-xTB stand-in). The model picks
which conformers to label by where its committee of models *disagrees most* — often the most
informative structures (though, as we'll see, not always).

### Experiment overview

1. Download a pretrained MACE organics model and finetune it on a few conformers of a molecule.
2. Use a committee (ensemble) of finetuned models to estimate prediction uncertainty.
3. Each round: propose perturbed conformers, pick the most uncertain ones, label them with the
   xTB oracle, and finetune again.
4. Compare an uncertainty-driven acquisition against a random baseline, and see why acquisition choice matters.

### Framework Components

1. **Dataset** (`ConformerDataset`, defined below): holds the molecule's conformers and their
   energies, and splits them into train/validation/test. The candidate pool is unused — new
   conformers are generated on the fly.
2. **Surrogate Model** ([`MLIPModel`](https://instadeepai.github.io/alf/api/alf_tools/models/)) wrapped in an [`EnsembleWrapper`](https://instadeepai.github.io/alf/api/alf_tools/models/) committee: each member finetunes the pretrained MACE model; their disagreement is our uncertainty.
3. **Search Strategy** ([`ProtocolSearch`](https://instadeepai.github.io/alf/api/alf_core/optimizer/search/) + `RattleSearch`): generates new conformers by perturbing current training structures.
4. **Acquisition Function** (`MaxVariance`, defined below): selects the conformers where the committee disagrees most (highest prediction variance). `RandomAcquisition` is the baseline for comparison.
5. **Optimizer** ([`Optimizer`](https://instadeepai.github.io/alf/api/alf_core/optimizer/optimizer/)): handles the ask/tell cycle.
6. **Oracle** ([`Oracle`](https://instadeepai.github.io/alf/api/alf_core/oracle/) + `XTBScorer`): computes ground-truth energy and forces with GFN2-xTB.
7. **Task** ([`DesignTask`](https://instadeepai.github.io/alf/api/alf_core/tasks/design_task/)): orchestrates the active-learning loop.

### Step 0: Environment Setup

Create the environment with `uv sync` from the `tutorials/` directory (CPU PyTorch by default).
This tutorial downloads a pretrained MACE organics model and the public **rMD17 aspirin** dataset
(both from Hugging Face, public, no credentials). The aspirin configurations carry DFT energies and
forces, so no live quantum-chemistry engine is needed.

In [ ]:
from pathlib import Path

from huggingface_hub import hf_hub_download, snapshot_download

# Pretrained MACE organics foundation model -> alf models dir, so the MLIPModel loader
# finds it locally and skips its (private) S3 fallback.
import alf_tools.models.utils.mlip_utils as mlip_utils  # noqa: E402

models_dir = mlip_utils._MODELS_DIR
models_dir.mkdir(parents=True, exist_ok=True)
hf_hub_download(
    repo_id="InstaDeepAI/mlip_models_organics_v2",
    filename="mace_organics_02.zip",
    local_dir=str(models_dir),
)

# Public rMD17 aspirin dataset (DFT energies + forces) from the mlip tutorials collection.
DATA_DIR = Path("data/aspirin")
snapshot_download(
    repo_id="InstaDeepAI/MLIP-tutorials",
    allow_patterns="training/rmd17_aspirin_*",
    local_dir=str(DATA_DIR),
)
ASPIRIN_TRAIN_XYZ = DATA_DIR / "training" / "rmd17_aspirin_train.xyz"

print(f"✅ Pretrained model present at {models_dir / 'mace_organics_02.zip'}")
print(f"✅ rMD17 aspirin dataset present at {ASPIRIN_TRAIN_XYZ}")

### Step 1: Import Required Libraries

In [ ]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from alf_core import (
    AcquisitionFunction,
    Candidate,
    DatasetSearch,
    DesignTask,
    FileStateLogger,
    LabelledCandidates,
    Optimizer,
    Oracle,
    State,
    Surrogate,
    TerminalStateLogger,
)
from alf_core.dataclasses.candidate import Modality
from alf_core.dataset.base_dataset import BaseDataset, BaseDatasetConfig
from alf_core.utils.enums import ProblemType
from alf_tools.models.ensemble import EnsembleWrapper, EnsembleWrapperConfig, SubsampleConfig
from alf_tools.models.mlip import MLIPModel, MLIPModelConfig, MLIPTrainConfig
from ase.io import read as ase_read

print("✅ All imports successful!")

### Step 2: Load the rMD17 Aspirin Dataset

We load configurations of **aspirin** — sampled from molecular-dynamics trajectories in the revised
MD17 (rMD17) dataset — from the public mlip tutorials collection. Each configuration carries a DFT
energy (our regression label) and forces (stored in each candidate's `features` so the MACE model
can train on them). We subsample `N_TOTAL` configurations and let ALF split them into a small
**seed-train** set, a **validation** set, a held-out **test** set, and a **candidate pool**. Active
learning then reveals labels from the pool a few configurations at a time.

In [ ]:
DATA_SEED = 51505
N_TOTAL = 150  # subsample of rMD17 aspirin configs to keep the notebook CPU-fast


def load_aspirin(xyz_path, max_configs=None, seed=0):
    """Read an extxyz file of aspirin configurations into `LabelledCandidates`.

    Energy (eV) is the regression label; forces (N, 3) are stored in each
    candidate's `features` so the MACE model can train on them.
    """
    atoms_list = ase_read(str(xyz_path), index=":")
    if max_configs is not None and len(atoms_list) > max_configs:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(atoms_list), size=max_configs, replace=False)
        atoms_list = [atoms_list[i] for i in idx]
    candidates, labels = [], []
    for atoms in atoms_list:
        energy = atoms.get_potential_energy()
        forces = atoms.get_forces()
        candidates.append(
            Candidate(data=atoms, modality=Modality.STRUCTURE, features={"forces": forces})
        )
        labels.append(energy)
    return LabelledCandidates(candidates=candidates, labels=np.asarray(labels))


class AspirinDataset(BaseDataset):
    """Pre-labelled aspirin conformers; alf splits them into seed-train / val / test / pool."""

    def __init__(self, config, data):
        """Store the pre-built labelled data; `setup()` performs the split."""
        super().__init__(config)
        self._data = data

    def load_dataset(self):
        """Return the pre-built `LabelledCandidates` (energy labels, forces in features)."""
        return self._data


def make_aspirin_dataset(seed=DATA_SEED):
    """Build and set up an `AspirinDataset` with the CPU-tiny split used in this tutorial."""
    data = load_aspirin(ASPIRIN_TRAIN_XYZ, max_configs=N_TOTAL, seed=seed)
    ds = AspirinDataset(
        BaseDatasetConfig(
            name="rmd17_aspirin",
            modality=Modality.STRUCTURE,
            seed=seed,
            train_ratio=0.12,
            validation_frac=0.2,
            test_ratio=0.25,
            split_type="random",
            problem_type=ProblemType.REGRESSION,
        ),
        data,
    )
    ds.setup()
    return ds


data = load_aspirin(ASPIRIN_TRAIN_XYZ, max_configs=N_TOTAL, seed=DATA_SEED)

# Fail fast if aspirin contains any element the pretrained organics model never saw.
from mlip.models.mace.network import Mace  # noqa: E402

_pretrained_ff = mlip_utils._load_model_from_zip(Mace, "mace_organics_02.zip")
_ztable = set(_pretrained_ff.dataset_info.atomic_energies_map.keys())
_missing = {int(n) for c in data.candidates for n in c.data.numbers} - _ztable
assert not _missing, f"aspirin contains elements outside the pretrained z-table: {_missing}"

dataset = make_aspirin_dataset()
print(dataset)
print(
    f"✅ rMD17 aspirin dataset ready — pool={len(dataset.candidate_pool)}, "
    f"test={len(dataset.test_dataset)}"
)

### Step 3: The Oracle (GFN2-xTB)

The oracle is the expensive ground-truth evaluator. In production this is DFT or a wet-lab
measurement; here we use GFN2-xTB — a fast, deterministic semi-empirical quantum method that
returns energy and forces. `XTBScorer` wraps it as an ALF `BaseModel`; when the `Oracle` evaluates
acquired conformers, it records both the energy (the label) and the forces (in `features`).

In [ ]:
class XTBScorer(BaseModel):
    """Oracle that scores conformers with GFN2-xTB, the 'expensive' ground truth."""

    def predict(self, candidate_points: list[Candidate]) -> Predictions:
        """Return xTB energies as means and store xTB forces on each candidate."""
        energies = []
        for cand in candidate_points:
            energy, forces = xtb_energy_forces(cand.data)
            if cand.features is None:
                cand.features = {}
            cand.features["forces"] = forces
            energies.append(energy)
        return Predictions(means=np.asarray(energies))

    def featurise(self, inputs):
        """No-op: the oracle does not featurise."""
        return inputs

    def train(self, train_data, val_data, metrics_collector=None):
        """No-op: the oracle is not trained."""
        return None

    def sample(self, condition=None):
        """Not implemented for the oracle."""
        raise NotImplementedError("XTBScorer does not sample.")


oracle = Oracle(scorer=XTBScorer())
print("✅ Oracle initialized (GFN2-xTB)!")

### Step 4: Search and Acquisition

`RattleSearch` proposes new candidate conformers by randomly perturbing the geometries of the
current training structures — the analogue of mutating a sequence. `MaxVariance` (used next) scores
them by committee disagreement, picking the conformers the ensemble is least sure about.
`RandomAcquisition` is the baseline that ignores the model and picks at random, so we can show that
uncertainty-driven selection actually helps.

In [ ]:
class RattleSearch(SearchProtocol):
    """Generate new conformers by rattling a random sample of training structures."""

    def __init__(self, n_seeds: int = 10, n_per_seed: int = 5, stdev: float = 0.06, seed: int = 0):
        """Configure how many seed structures to perturb and how strongly."""
        self.n_seeds = n_seeds
        self.n_per_seed = n_per_seed
        self.stdev = stdev
        self.seed = seed

    def __call__(self, state: State) -> list[Candidate]:
        """Return rattled copies of randomly chosen current training conformers."""
        train = state.dataset.train_dataset
        rng = np.random.default_rng(self.seed)
        n_seeds = min(self.n_seeds, len(train))
        seed_idx = rng.choice(len(train), size=n_seeds, replace=False)
        pool: list[Candidate] = []
        counter = 0
        for idx in seed_idx:
            base = train.candidates[int(idx)].data
            for _ in range(self.n_per_seed):
                a = base.copy()
                a.rattle(stdev=self.stdev, seed=self.seed + counter)
                pool.append(Candidate(data=a, modality=Modality.STRUCTURE))
                counter += 1
        return pool


class MaxVariance(AcquisitionFunction):
    """Select conformers where the committee disagrees most (highest prediction variance).

    This is query-by-committee uncertainty sampling: the structures the ensemble is least
    certain about are the most informative to label next. It is the natural acquisition for
    *reducing model error* (our goal), as opposed to maximising a property.
    """

    def __call__(self, search_candidates: list[Candidate], state: State) -> LabelledCandidates:
        """Score each candidate by its committee variance (higher = more uncertain)."""
        predictions = state.surrogate.predict(search_candidates)
        if predictions.variances is None:
            raise ValueError("MaxVariance requires committee variances; use an ensemble surrogate.")
        return LabelledCandidates(candidates=search_candidates, labels=predictions.variances)


class RandomAcquisition(AcquisitionFunction):
    """Baseline acquisition that scores candidates uniformly at random."""

    def __init__(self, seed: int = 0):
        """Seed the RNG used to score candidates."""
        self.seed = seed

    def __call__(self, search_candidates: list[Candidate], state: State) -> LabelledCandidates:
        """Assign each candidate a random acquisition value."""
        rng = np.random.default_rng(self.seed)
        scores = rng.random(len(search_candidates))
        return LabelledCandidates(candidates=search_candidates, labels=scores)


search_fn = ProtocolSearch(protocol=RattleSearch(n_seeds=10, n_per_seed=5, stdev=0.06, seed=0))
print("✅ Search strategy initialized (rattle perturbations)!")

### Step 5: The Surrogate — a Committee of Finetuned MACE Models

Our surrogate is an ensemble (committee) of `MLIPModel`s. Each member finetunes the **same**
pretrained MACE model, but on a different bootstrap resample of the training data, so the members
end up slightly different. Where they disagree most, the model is most uncertain — exactly the
conformers worth labelling. `EnsembleWrapper.predict` returns both the mean energy and the
variance across members.

In [ ]:
def mlip_factory(seed: int) -> MLIPModel:
    """Build a finetuning MLIPModel (from the pretrained MACE model) with the given seed."""
    return MLIPModel(
        model_config=MLIPModelConfig(model_path="mace_organics_02.zip"),
        train_config=MLIPTrainConfig(epochs=8, batch_size=2, learning_rate=1e-3),
        seed=seed,
    )


def make_surrogate(n_members: int = 2) -> Surrogate:
    """Build a committee surrogate of finetuned MACE models (1 member = no uncertainty)."""
    return Surrogate(
        model=EnsembleWrapper(
            model_factory=mlip_factory,
            config=EnsembleWrapperConfig(
                base_seed=0,
                n_members=n_members,
                subsample=SubsampleConfig(fraction=1.0, replace=True),
            ),
        )
    )


surrogate = make_surrogate(n_members=2)
print("✅ Surrogate committee initialized (2 finetuned MACE members)!")

### Step 6: Optimizer and Design Task

`MaxVariance` ranks candidates purely by the committee's prediction variance, so each round we
label the conformers the ensemble disagrees on most — classic query-by-committee active learning.
The `Optimizer` combines it with the rattle search; `DesignTask` runs the rounds.

In [ ]:
acquisition_fn = MaxVariance()
optimizer = Optimizer(acquisition_fn=acquisition_fn, search_fn=search_fn)

num_acq_rounds = 2
acq_batch_size = 3
task = DesignTask(num_acq_rounds=num_acq_rounds, acq_batch_size=acq_batch_size)
print(f"✅ Optimizer + DesignTask ready ({num_acq_rounds} rounds × {acq_batch_size} labels)!")

### Step 7: Run the Active-Learning Experiment (uncertainty-driven)

Each round: propose conformers, pick the 5 most uncertain, label them with xTB, finetune the
committee, and evaluate on the held-out test set.

In [ ]:
import logging

logging.basicConfig(level=logging.WARNING)  # keep MACE/xTB output quiet in the notebook

maxvar_path = Path("results/mlip_design/")
if maxvar_path.exists():
    shutil.rmtree(maxvar_path)
loggers = [TerminalStateLogger(), FileStateLogger(output_path=maxvar_path)]

state = task.setup(dataset=dataset, surrogate=surrogate)
print("🚀 Running uncertainty-driven active learning...")
task.run(state, state_loggers=loggers, optimizer=optimizer, oracle=oracle)
print("✅ Uncertainty-driven experiment completed!")

### Step 8: Random-Acquisition Baseline

To show that uncertainty-driven selection helps, we run the identical loop but pick conformers at
random. The baseline needs no committee, so it uses a single finetuned model.

In [ ]:
# Rebuild a fresh dataset so the baseline starts from the same initial splits.
dataset_random = ConformerDataset(
    BaseDatasetConfig(
        name="ethanol_conformers",
        modality=Modality.STRUCTURE,
        seed=51505,
        train_ratio=0.4,
        validation_frac=0.2,
        test_ratio=0.3,
        split_type="random",
        problem_type=ProblemType.REGRESSION,
    ),
    build_labelled(conformers),
)
dataset_random.setup()

random_optimizer = Optimizer(acquisition_fn=RandomAcquisition(seed=0), search_fn=search_fn)
random_surrogate = make_surrogate(n_members=1)  # no uncertainty needed for random
random_task = DesignTask(num_acq_rounds=num_acq_rounds, acq_batch_size=acq_batch_size)

random_path = Path("results/mlip_design_random/")
if random_path.exists():
    shutil.rmtree(random_path)
random_loggers = [TerminalStateLogger(), FileStateLogger(output_path=random_path)]

random_state = random_task.setup(dataset=dataset_random, surrogate=random_surrogate)
print("🚀 Running random-acquisition baseline...")
random_task.run(
    random_state, state_loggers=random_loggers, optimizer=random_optimizer, oracle=oracle
)
print("✅ Baseline completed!")

### Step 9: Results

The headline metric is the **learning curve**: test-set energy error against the number of labels
acquired. If uncertainty-driven acquisition works, its error drops faster than the random
baseline's. We also show a parity plot of the finetuned baseline model's predicted vs. xTB energies on the test
set.

In [ ]:
maxvar_metrics = pd.read_csv("results/mlip_design/metrics.csv")
rnd_metrics = pd.read_csv("results/mlip_design_random/metrics.csv")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("MLIP Active Learning: uncertainty-driven vs random", fontsize=15, fontweight="bold")

for df, label, color in [
    (maxvar_metrics, "MaxVariance committee", "#e74c3c"),
    (rnd_metrics, "Random", "#3498db"),
]:
    axes[0].plot(
        df["dataset/num_train"], df["surrogate/test_mse"], marker="o", label=label, color=color
    )
    axes[1].plot(
        df["dataset/num_train"], df["surrogate/test_spearman"], marker="o", label=label, color=color
    )

axes[0].set_xlabel("Number of labelled structures")
axes[0].set_ylabel("Test energy MSE (eV²)")
axes[0].set_title("Learning curve (lower is better)")
axes[0].grid(True, alpha=0.3)
axes[0].legend()

axes[1].set_xlabel("Number of labelled structures")
axes[1].set_ylabel("Test Spearman")
axes[1].set_title("Rank correlation (higher is better)")
axes[1].grid(True, alpha=0.3)
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Parity on the held-out test set. We use the (stable) random-baseline model — the
# MaxVariance run over-acquires distorted conformers and is less reliable here.
test_cands = dataset_random.test_dataset.candidates
test_true = dataset_random.test_dataset.labels
test_pred = random_surrogate.predict(test_cands).means

plt.figure(figsize=(6, 6))
plt.scatter(test_true, test_pred, alpha=0.7, color="#9b59b6")
lims = [min(test_true.min(), test_pred.min()), max(test_true.max(), test_pred.max())]
plt.plot(lims, lims, "k--", alpha=0.5)
plt.xlabel("xTB energy (eV)")
plt.ylabel("Predicted energy (eV)")
plt.title("Finetuned MLIP: predicted vs. xTB energy (test set)")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Conclusion

We built an online active-learning loop for a machine-learned interatomic potential with ALF:
finetune a pretrained MACE committee on a few xTB-labelled conformers, estimate uncertainty from
committee disagreement, acquire new conformers, and repeat. Every component is a standard ALF
abstraction — `ConformerDataset`, the `XTBScorer` oracle, `RattleSearch`, the `EnsembleWrapper`
surrogate, `MaxVariance`/`RandomAcquisition`, `Optimizer`, and `DesignTask`.

This example is deliberately tiny and fast, so treat the learning curves as illustrative rather
than conclusive. On a problem this small, uncertainty (max-variance) sampling can over-acquire the
most distorted conformers and does not reliably beat random selection — a useful reminder that the
acquisition strategy must be matched to the problem and validated, not assumed. Query-by-committee
active learning is the workhorse of *production* MLIP training, where it selects from large,
diverse pools of physically meaningful structures.

### Next steps

- Scale up: more initial data, a larger curated candidate pool, and a bigger held-out test set.
- Constrain how far acquired conformers stray from equilibrium so finetuning does not overfit outliers.
- Swap the xTB oracle for DFT, or train from scratch (`model_path=None`) for chemistries with no
  compatible foundation model.

**Happy modelling!** ⚛️🔬✨

In [ ]:
for p in [Path("results/mlip_design/"), Path("results/mlip_design_random/")]:
    if p.exists():
        shutil.rmtree(p)
print("✅ Results directories cleaned up!")